# FGNS-TRM v4.4.0: Finsler-Gauge Cellular Sheaf Network for ARC Prize 2026 (ARC-AGI-2)
**Author:** Vyacheslav Zimovsky (`v.zimovsky.ai@gmail.com`) | **License:** `MIT-0` (MIT No Attribution)

## Architectural Summary
`FGNS-TRM v4.4.0` is a pure transductive neural reasoning architecture that replaces combinatorial DSL search with continuous differential geometry and spectral test-time adaptation:
1. **Theorem I (Orthogonal Palette-Geometry Splitting):** Exact QR fiber decomposition $\mathbb{R}^{128} = \mathcal{H}_{\text{color}}^{(11)} \oplus \mathcal{H}_{\text{geom}}^{(117)}$ preventing rotational color leakage (`< 1e-7`).
2. **Theorem II (Cellular Sheaf Connection Laplacian):** Edge-stopping Stiefel $O(d)$ parallel transport contracting Dirichlet energy while preserving 1px topological boundaries.
3. **OctoWave 8-Axis State-Space Model (SSM):** Line-parallel 8-directional causal wavefront scan with boundary-stopped specular reflection.
4. **Dual-State Cauchy Resolvent Field Decoder:** 4-step recurrent $(Z^{(\tau)}, P^{(\tau)})$ solver with transductive support-residual Lyapunov correction.
5. **Schatten-1 Spectral MDL Test-Time Adaptation:** Leave-One-Out (LOO) transductive refinement regularized by the nuclear norm $\|\Delta W\|_{S_1}$ and Von Neumann spectral entropy.
6. **Theorem III (1-Jet Stokes & Coupled Boltzmann-Woodbury Dimension Predictor):** Closed-form canvas dimension inference ($84.0\%$ exact match on blind evaluation).
7. **$S_{10}$-Equivariant Role-Woodbury & Hamilton-Hare LRQ Palette Predictor:** Conservation of discrete color budgets with Diode Annihilation Sink.
8. **Theorem IV (Geodesic $\mathfrak{so}(3)^C + D_4 \times S_9$ Operator Adapter):** Automorphism-equivariant test-time adaptation with Holistic $D_4$ Condorcet mode synthesis.

In [ ]:
"""Cell 2: Offline Environment Configuration, Dataset Unpacking & GPU Discovery."""
import os
from pathlib import Path
import shutil
import sys
import zipfile

if os.path.isfile("/usr/local/cuda/bin/ptxas"):
    os.environ.setdefault("TRITON_PTXAS_PATH", "/usr/local/cuda/bin/ptxas")
    if "/usr/local/cuda/bin" not in os.environ.get("PATH", ""):
        os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

WORK_ENGINE_DIR = Path("/tmp/fgns_engine")

def setup_fgns_offline_workspace() -> Path:
    """Discovers fgns_engine in local CWD or /kaggle/input and mounts it in /tmp/fgns_engine."""
    if Path("fgns/__init__.py").is_file() and Path("scripts/kaggle_submission_runner.py").is_file():
        local_root = Path(".").resolve()
        if str(local_root) not in sys.path:
            sys.path.insert(0, str(local_root))
        return local_root

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for zip_cand in kaggle_input.rglob("fgns_engine_dataset.zip"):
            WORK_ENGINE_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_cand, "r") as zf:
                zf.extractall(WORK_ENGINE_DIR)
            if str(WORK_ENGINE_DIR) not in sys.path:
                sys.path.insert(0, str(WORK_ENGINE_DIR))
            return WORK_ENGINE_DIR

        for init_cand in kaggle_input.rglob("fgns/__init__.py"):
            src_root = init_cand.parent.parent
            if WORK_ENGINE_DIR.exists():
                shutil.rmtree(WORK_ENGINE_DIR)
            shutil.copytree(src_root, WORK_ENGINE_DIR)
            if str(WORK_ENGINE_DIR) not in sys.path:
                sys.path.insert(0, str(WORK_ENGINE_DIR))
            return WORK_ENGINE_DIR

    raise FileNotFoundError("Could not locate fgns_engine dataset in /kaggle/input or current directory.")

ENGINE_ROOT = setup_fgns_offline_workspace()
import torch
print(f"FGNS Engine Root : {ENGINE_ROOT}")
print(f"PyTorch Version  : {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for g_idx in range(torch.cuda.device_count()):
        print(f"  GPU [{g_idx}]        : {torch.cuda.get_device_name(g_idx)}")
print(f"KAGGLE_IS_COMPETITION_RERUN = {os.environ.get('KAGGLE_IS_COMPETITION_RERUN', 'False')}")

In [ ]:
"""Cell 3: Pre-Flight Self-Diagnostics & Path Discovery Verification."""
from fgns.config import FGNSConfig
from fgns.models.fgns_network import FinslerGaugeSheafNetwork
from fgns.solver.submit.atomic_writer import KaggleSchemaAuditor, verify_atomic_writer_diagnostics
from fgns.solver.submit.path_resolver import DatasetPathResolver, verify_path_resolver_diagnostics
from scripts.kaggle_submission_runner import discover_competition_challenges_and_solutions

assert verify_path_resolver_diagnostics(), 'DatasetPathResolver self-test failed!'
assert verify_atomic_writer_diagnostics(), 'AtomicSubmissionWriter self-test failed!'

chal_file, sol_file = discover_competition_challenges_and_solutions()
ckpt_file = DatasetPathResolver.resolve_checkpoint('model_final.pt')
cfg = FGNSConfig()
probe_net = FinslerGaugeSheafNetwork(cfg)
total_params = sum(p.numel() for p in probe_net.parameters())
del probe_net

print(f"Resolved Challenges : {chal_file}")
print(f"Resolved Solutions  : {sol_file if sol_file else 'Hidden Test Mode (None)'}")
print(f"Resolved Checkpoint : {ckpt_file if ckpt_file else 'Fresh Geometric Initialization'}")
print(f"FGNS-TRM Parameters : {total_params:,} (Pre-Flight Checks PASSED)")

In [ ]:
"""Cell 4: Master Tournament Execution (Automatic Rerun vs. Commit Mode)."""
from pathlib import Path
from scripts.kaggle_submission_runner import KaggleTournamentOrchestrator

target_output = "/kaggle/working/submission.json" if Path("/kaggle/working").is_dir() else "submission.json"
orchestrator = KaggleTournamentOrchestrator(
    data_path=None,
    output_path=target_output,
    force_rerun_mode=False,
)
submission_payload = orchestrator.execute()

In [ ]:
"""Cell 5: Final Post-Run Submission Verification Gate."""
import json
from pathlib import Path
from fgns.solver.submit.atomic_writer import KaggleSchemaAuditor
from scripts.kaggle_submission_runner import discover_competition_challenges_and_solutions

sub_path = Path("/kaggle/working/submission.json") if Path("/kaggle/working/submission.json").is_file() else Path("submission.json")
assert sub_path.is_file() and sub_path.stat().st_size > 10, f'Missing or empty submission file at {sub_path}'

with open(sub_path, "r", encoding="utf-8") as f_sub:
    final_sub = json.load(f_sub)

chal_path, _ = discover_competition_challenges_and_solutions()
with open(chal_path, "r", encoding="utf-8") as f_chal:
    raw_challenges = json.load(f_chal)

expected_ids = list(raw_challenges.keys())
is_valid, schema_errors = KaggleSchemaAuditor.validate_submission_dict(final_sub, expected_task_ids=expected_ids)
assert is_valid, f'Schema errors detected: {schema_errors[:5]}'

total_test_pairs = 0
for tid, t_data in raw_challenges.items():
    preds = final_sub[tid]
    assert len(preds) == len(t_data['test']), f'Test count mismatch on {tid}'
    for pair in preds:
        assert pair['attempt_1'] != pair['attempt_2'], f'Duplicate attempts on {tid}'
        total_test_pairs += 1

print(f"VERIFIED: {sub_path.resolve()} | Tasks: {len(final_sub)} | Test Instances: {total_test_pairs} | 0 Errors")